In [30]:
!pip list | grep scikit-learn

scikit-learn                       1.5.1


In [31]:
!python -V

Python 3.12.7


In [32]:
import pickle
import pandas as pd

In [33]:
with open('/Users/muhammadwisalabdullah/PycharmProjects/MLOPsZoomcampHWs/HW1/mlruns/1/1e6ca94c7943433f9540263ca2dc4f34/artifacts/dict_vectorizer.bin', "rb") as f_in:
    dv = pickle.load(f_in)

In [34]:
with open('/Users/muhammadwisalabdullah/PycharmProjects/MLOPsZoomcampHWs/HW1/mlruns/1/1e6ca94c7943433f9540263ca2dc4f34/artifacts/model/model.pkl', 'rb') as f_in:
    model = pickle.load(f_in)

In [35]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)

    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')

    return df

In [36]:
train_df = read_data('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet')

#categorical = ['PULocationID', 'DOLocationID']
train_df['PU_DO'] = train_df['PULocationID'] + '_' + train_df['DOLocationID']
categorical = ['PU_DO']
train_dicts = train_df[categorical].to_dict(orient='records')


In [37]:
X_train = dv.transform(train_dicts)
y_pred = model.predict(X_train)

In [38]:
print(y_pred)

[ 6.29068441 44.15786134 15.87085074 ... 15.20085986 11.9399909
 13.52263918]


In [39]:
import numpy as np

y_pred = model.predict(X_train)
std_dev = np.std(y_pred)

print(f"Standard deviation of predictions: {std_dev:.4f}")

Standard deviation of predictions: 8.4181


In [40]:
year = 2023
month = 3

In [41]:
output_file = "/Users/muhammadwisalabdullah/PycharmProjects/MLOPsZoomcampHWs/HW4/results_df.parquet"

# Must be defined before use

In [42]:
train_df['ride_id'] = f'{year:04d}/{month:02d}_' + train_df.index.astype('str')

In [43]:
df_result = pd.DataFrame()
df_result['ride_id'] = train_df['ride_id']
df_result['predicted_duration'] = y_pred

import os
os.makedirs("/Users/muhammadwisalabdullah/PycharmProjects/MLOPsZoomcampHWs/HW4/", exist_ok=True)

In [44]:
train_df.to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False
)